# Лабораторная работа: Рекомендательные системы

## Теоретическая часть

### 1. Суть задачи рекомендательных систем
Рекомендательные системы – это алгоритмы, которые анализируют поведение пользователей и предлагают персонализированные рекомендации товаров, фильмов, музыки и других объектов. Основная цель – предсказать предпочтения пользователей на основе имеющихся данных о взаимодействиях.


### 2. Метод коллаборативной фильтрации
Коллаборативная фильтрация (Collaborative Filtering, CF) – это метод рекомендаций, основанный на анализе поведения пользователей. Он работает на основе предположения, что пользователи с похожими предпочтениями в прошлом будут делать схожий выбор в будущем.

Существует два основных подхода:
1. **User-based CF** – рекомендации строятся на основе сходства пользователей.
2. **Item-based CF** – рекомендации строятся на основе сходства объектов.

### 3. Латентные факторные модели (Matrix Factorization)
Коллаборативная фильтрация может быть реализована через матричное разложение. Пусть у нас есть матрица взаимодействий пользователей и объектов R, где $( R_{u,i} )$ – оценка пользователя ( u ) для объекта ( i ). Тогда разложение можно представить в виде:
$$
R \approx U \cdot V^T
$$
где:
- ( U ) – матрица эмбеддингов пользователей,
- ( V ) – матрица эмбеддингов объектов.

Предсказание рейтинга рассчитывается как:
$$
\hat{R}_{u,i} = U_u \cdot V_i^T
$$

В данной лабораторной работе предполагается использование **нейросетевого метода**, который обучает эмбеддинги пользователей и объектов с помощью полносвязных слоев. Входные данные – индексы пользователей и объектов, которые преобразуются в векторные представления, а затем подаются на вход нейросети.


## Практическая часть
В данной работе вам предлагается реализовать рекомендательную систему на основе метода коллаборативной фильтрации, используя нейросетевую модель. Вы должны:
1. Подготовить данные: загрузить свой датасет (например, рейтинг фильмов, товаров, книг и т. д.).
2. Разбить данные на тренировочный и тестовый наборы.
3. Обучить модель, используя эмбеддинги пользователей и объектов.
4. Оценить качество модели на тестовом наборе.
5. Вывести список рекомендаций для выбранного пользователя.

In [17]:
# Импорты
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Определяем устройство (используем GPU, если доступно)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cpu


In [5]:
# Загрузка датасета
# !wget http://files.grouplens.org/datasets/movielens/ml-100k/u.data -O ratings.csv

In [6]:
# Определяем названия столбцов
columns = ['user_id', 'anime_id', 'rating']
df = pd.read_csv('rating_complete.csv', sep='\t')

In [7]:
df = pd.read_csv('rating_complete.csv', sep=',')

df = df[df['rating'] != -1]
df = df.head(10_000)

df['user_id'] = pd.factorize(df['user_id'])[0]
df['anime_id'] = pd.factorize(df['anime_id'])[0]

print("NaN в rating:", df['rating'].isna().sum())
df.info()

NaN в rating: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   user_id   10000 non-null  int64
 1   anime_id  10000 non-null  int64
 2   rating    10000 non-null  int64
dtypes: int64(3)
memory usage: 234.5 KB


In [8]:
# Определяем датасет PyTorch
class RatingsDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df['user_id'].values, dtype=torch.long)
        self.items = torch.tensor(df['anime_id'].values, dtype=torch.long)
        self.ratings = torch.tensor(df['rating'].values, dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.ratings[idx]

In [9]:
# Определяем нейросетевую модель для коллаборативной фильтрации
class RecommenderNN(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=32):
        super(RecommenderNN, self).__init__()
        # Эмбеддинги пользователей и фильмов
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)

        # Полносвязные слои для предсказания рейтинга
        self.fc_layers = nn.Sequential(
            nn.Linear(embedding_dim * 2, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, user, item):
        # Получаем эмбеддинги пользователя и фильма
        user_emb = self.user_embedding(user)
        item_emb = self.item_embedding(item)

        # Объединяем эмбеддинги
        x = torch.cat([user_emb, item_emb], dim=1)

        # Пропускаем через полносвязные слои
        return self.fc_layers(x).squeeze()

In [10]:
# Определяем количество пользователей и фильмов
num_users = df['user_id'].nunique()
num_items = df['anime_id'].nunique()


In [11]:
# Создаём датасеты и загрузчики данных
dataset = RatingsDataset(df)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

In [12]:
# Инициализация модели
model = RecommenderNN(num_users, num_items).to(device)

# Определяем функцию потерь (MSE) и оптимизатор (Adam)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)


In [19]:
for epoch in range(30):
    model.train()
    total_loss = 0
    all_predictions = []
    all_ratings = []
    for users, items, ratings in train_loader:
        users, items, ratings = users.to(device), items.to(device), ratings.to(device)
        optimizer.zero_grad()
        predictions = model(users, items)
        loss = criterion(predictions, ratings)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        all_predictions.extend(predictions.cpu().detach().numpy())
        all_ratings.extend(ratings.cpu().detach().numpy())

    # Средняя ошибка предсказания на тренировочной выборке
    rmse = math.sqrt(mean_squared_error(all_ratings, all_predictions))
    mae = mean_absolute_error(all_ratings, all_predictions)

    print(f'Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}')

# Оценка модели на тестовом наборе
model.eval()
test_predictions = []
test_ratings = []
with torch.no_grad():
    for users, items, ratings in test_loader:
        users, items, ratings = users.to(device), items.to(device), ratings.to(device)
        predictions = model(users, items)
        test_predictions.extend(predictions.cpu().numpy())
        test_ratings.extend(ratings.cpu().numpy())

# Средняя ошибка на тестовом наборе
test_rmse = math.sqrt(mean_squared_error(test_ratings, test_predictions))
test_mae = mean_absolute_error(test_ratings, test_predictions)

print(f'\nTest RMSE: {test_rmse:.4f}, Test MAE: {test_mae:.4f}')

# Рекомендации для нескольких случайных пользователей
random_users = np.random.choice(df['user_id'].unique(), size=5)

print("\nRecommendations for random users:")
for user_id in random_users:
    # Предсказания для всех объектов для выбранного пользователя
    user_tensor = torch.tensor([user_id] * num_items, dtype=torch.long).to(device)
    item_tensor = torch.tensor(range(num_items), dtype=torch.long).to(device)

    with torch.no_grad():
        predictions = model(user_tensor, item_tensor).cpu().numpy()

    # Выбираем топ-5 рекомендованных объектов
    top_items = predictions.argsort()[-5:][::-1]

    print(f"User {user_id + 1}: Recommended items {top_items + 1}")

Epoch 1, Loss: 0.2099569825530052
Epoch 2, Loss: 0.19234302937984465
Epoch 3, Loss: 0.18008101254701614
Epoch 4, Loss: 0.16173204165697097
Epoch 5, Loss: 0.1586886031627655
Epoch 6, Loss: 0.14530420619249343
Epoch 7, Loss: 0.1422950986623764
Epoch 8, Loss: 0.1397132551074028
Epoch 9, Loss: 0.13027743732929228
Epoch 10, Loss: 0.12024788761138916
Epoch 11, Loss: 0.12078604462742805
Epoch 12, Loss: 0.1183218184709549
Epoch 13, Loss: 0.10110420978069305
Epoch 14, Loss: 0.09508300268650055
Epoch 15, Loss: 0.08825261250138283
Epoch 16, Loss: 0.07968848589062691
Epoch 17, Loss: 0.07926291951537132
Epoch 18, Loss: 0.07609723335504531
Epoch 19, Loss: 0.08889069125056266
Epoch 20, Loss: 0.08899811378121376
Epoch 21, Loss: 0.08339729261398315
Epoch 22, Loss: 0.09091910555958747
Epoch 23, Loss: 0.08260199999809265
Epoch 24, Loss: 0.07330065715312958
Epoch 25, Loss: 0.07685659831762313
Epoch 26, Loss: 0.07077676528692245
Epoch 27, Loss: 0.06363363760709763
Epoch 28, Loss: 0.06016983500123024
Epoch 

## Реализация функции определения топ N аниме для пользователя с выбранным идентификатором id.

Подбор рекомендаций осуществляется на основе предсказанных моделью рейтингов. Для каждого объекта, с которым пользователь ещё не взаимодействовал, вычисляется ожидаемое значение рейтинга. Далее формируется список Top-N объектов с наибольшими предсказанными значениями.

In [20]:
def top_n_anime_for_user(model, df, user_id, num_items, N=5):
    model.eval()
    
    # Оцененные аниме
    watched_items = df[df['user_id'] == user_id]['anime_id'].values
    
    # Все аниме
    all_items = np.arange(num_items)
    
    # Убираем просмотренные
    items_to_predict = np.setdiff1d(all_items, watched_items)
    
    user_tensor = torch.tensor([user_id] * len(items_to_predict), dtype=torch.long).to(device)
    item_tensor = torch.tensor(items_to_predict, dtype=torch.long).to(device)
    
    with torch.no_grad():
        predictions = model(user_tensor, item_tensor).cpu().numpy()
    
    # Топ-N
    top_indices = predictions.argsort()[-N:][::-1]
    top_items = items_to_predict[top_indices]
    
    return top_items, predictions[top_indices]

In [ ]:
user_id = 10
top_items, scores = top_n_anime_for_user(model, df, user_id, num_items, N=5)

print(f"Top-5 рекомендаций для пользователя {user_id}:")
for item, score in zip(top_items, scores):
    print(f"Аниме ID: {item}, предсказанный рейтинг: {score:.2f}")

Top-5 рекомендаций для пользователя 10:
Аниме ID: 236, предсказанный рейтинг: 12.26
Аниме ID: 1613, предсказанный рейтинг: 12.18
Аниме ID: 737, предсказанный рейтинг: 12.15
Аниме ID: 2853, предсказанный рейтинг: 12.08
Аниме ID: 559, предсказанный рейтинг: 12.02


Из вывода видно, что предсказанный рейтинг может заходить за границу наибольшой возможной оценки - 10. Это обусловлено архитектурой составленной модели. Предсказанные моделью рейтинги могут выходить за пределы допустимого диапазона, так как выходной слой нейронной сети является линейным и не накладывает ограничений на диапазон значений. Для корректной интерпретации результатов можно ограничить выход модели, например, с помощью сигмоидальной функции активации и масштабирования значений к диапазону допустимых значений.

## Анализ работы рекомендательной системы и пути улучшения модели

Построенная модель представляет собой модель коллаборативной фильтрации. Модель использует обучаемые векторные представления (эмбеддинги) пользователей и аниме, которые объединяются и подаются на вход полносвязной нейронной сети для предсказания рейтингов.

### Ключевые этапы работы системы

1. Подготовка данных.
Были загружены данные (10000 записей) о взаимодействиях пользователей с объектами, выполнена очистка: удаление записей, содержащих пустые значения и значения не подходящие для обработки (аниме  с рейтингом -1), категориальные идентификаторы пользователей и объектов преобразованы в числовые индексы. Далее данные были разделены на обучающую и тестовую выборки в соотношении 80/20.

2. Построение и обучение модели.
Для пользователей и объектов были обучены эмбеддинги фиксированной размерности. После конкатенации эмбеддингов применялась полносвязная нейросеть, обучаемая с использованием функции потерь MSE и оптимизатора Adam. Обучение проводилось в течение 30 эпох с контролем значения функции потерь.

3. Оценка качества модели.
Качество работы системы оценивалось на тестовой выборке с использованием метрик RMSE и MAE. Полученные значения (RMSE = 1.66, MAE = 1.27) свидетельствуют о том, что модель в целом способна улавливать предпочтения пользователей, однако точность предсказаний ограничена объёмом данных и простотой архитектуры.

4. Генерация рекомендаций.
Для заданного пользователя формир список Top-N рекомендаций на основе максимальных предсказанных рейтингов для объектов, с которыми пользователь ранее не взаимодействовал. Это позволяет получать персонализированные рекомендации.

### Анализ результатов

Модель демонстрирует устойчивое уменьшение значения функции потерь на обучающей выборке и приемлемые значения RMSE и MAE на тестовой. Однако было выявлено, что предсказанные рейтинги могут выходить за пределы допустимого диапазона (например, превышать 10), что связано с использованием линейного выходного слоя без ограничения диапазона значений. Это не влияет на процесс ранжирования объектов, но ухудшает интерпретируемость предсказаний.

Кроме того, точность модели ограничена:
- относительно небольшим объёмом используемых данных
- отсутствием регуляризации
- использованием фиксированных гиперпараметров без подбора

### Возможные пути улучшения модели (изменение гиперпараметров и архитектуры)

1. Изменение размерности эмбеддингов.
Увеличение размерности эмбеддингов пользователей и объектов (например, с 32 до 64 или 128) может позволить модели захватывать более сложные зависимости между пользователями и объектами.

2. Подбор скорости обучения (learning rate).
Уменьшение шага обучения (например, с 0.005 до 0.001) может сделать процесс обучения более стабильным и снизить риск переобучения.

3. Увеличение числа эпох обучения.
Увеличение числа эпох (например, до 30–50) позволит модели лучше аппроксимировать зависимости в данных при условии контроля переобучения.

4. Добавление регуляризации.
Использование Dropout в полносвязных слоях и weight decay в оптимизаторе может снизить переобучение и улучшить обобщающую способность модели.

5. Расширение архитектуры модели.
Увеличение числа скрытых слоёв или нейронов может повысить выразительную способность модели, однако требует аккуратного подбора гиперпараметров для предотвращения переобучения.

## Улучшение модели

In [ ]:
class TunedRecommenderNN(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=64, dropout=0.1):
        super().__init__()
        
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)

        self.fc_layers = nn.Sequential(
            nn.Linear(embedding_dim * 2, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, user, item):
        user_emb = self.user_embedding(user)
        item_emb = self.item_embedding(item)
        x = torch.cat([user_emb, item_emb], dim=1)
        return self.fc_layers(x).squeeze()

В улучшенной версии модели были увеличены размерности эмбеддингов и добавлена регуляризация с помощью Dropout. Эти изменения направлены на снижение переобучения и повышение интерпретируемости предсказаний рейтингов.

In [32]:
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

In [33]:
# Инициализация модели
tuned_model = TunedRecommenderNN(num_users, num_items).to(device)

# Определяем функцию потерь (MSE) и оптимизатор (Adam)
criterion = nn.MSELoss()
optimizer = optim.Adam(tuned_model.parameters(), lr=0.005)


In [34]:
for epoch in range(30):
    tuned_model.train()
    total_loss = 0
    all_predictions = []
    all_ratings = []
    for users, items, ratings in train_loader:
        users, items, ratings = users.to(device), items.to(device), ratings.to(device)
        optimizer.zero_grad()
        predictions = tuned_model(users, items)
        loss = criterion(predictions, ratings)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        all_predictions.extend(predictions.cpu().detach().numpy())
        all_ratings.extend(ratings.cpu().detach().numpy())

    # Средняя ошибка предсказания на тренировочной выборке
    rmse = math.sqrt(mean_squared_error(all_ratings, all_predictions))
    mae = mean_absolute_error(all_ratings, all_predictions)

    print(f'Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}')

# Оценка модели на тестовом наборе
tuned_model.eval()
test_predictions = []
test_ratings = []
with torch.no_grad():
    for users, items, ratings in test_loader:
        users, items, ratings = users.to(device), items.to(device), ratings.to(device)
        predictions = tuned_model(users, items)
        test_predictions.extend(predictions.cpu().numpy())
        test_ratings.extend(ratings.cpu().numpy())

# Средняя ошибка на тестовом наборе
test_rmse = math.sqrt(mean_squared_error(test_ratings, test_predictions))
test_mae = mean_absolute_error(test_ratings, test_predictions)

print(f'\nTest RMSE: {test_rmse:.4f}, Test MAE: {test_mae:.4f}')

# Рекомендации для нескольких случайных пользователей
random_users = np.random.choice(df['user_id'].unique(), size=5)

print("\nRecommendations for random users:")
for user_id in random_users:
    # Предсказания для всех объектов для выбранного пользователя
    user_tensor = torch.tensor([user_id] * num_items, dtype=torch.long).to(device)
    item_tensor = torch.tensor(range(num_items), dtype=torch.long).to(device)

    with torch.no_grad():
        predictions = tuned_model(user_tensor, item_tensor).cpu().numpy()

    # Выбираем топ-5 рекомендованных объектов
    top_items = predictions.argsort()[-5:][::-1]

    print(f"User {user_id + 1}: Recommended items {top_items + 1}")

Epoch 1, Loss: 6.07116888999939
Epoch 2, Loss: 2.4715051364898684
Epoch 3, Loss: 2.1328312425613403
Epoch 4, Loss: 1.9109270877838134
Epoch 5, Loss: 1.6373098502159118
Epoch 6, Loss: 1.5525800576210023
Epoch 7, Loss: 1.3577650737762452
Epoch 8, Loss: 1.237408441543579
Epoch 9, Loss: 1.143746169090271
Epoch 10, Loss: 1.0574627661705016
Epoch 11, Loss: 0.9786517071723938
Epoch 12, Loss: 0.9150992205142975
Epoch 13, Loss: 0.8733968706130981
Epoch 14, Loss: 0.8433712592124939
Epoch 15, Loss: 0.8048574886322022
Epoch 16, Loss: 0.7940985910892486
Epoch 17, Loss: 0.7327013051509857
Epoch 18, Loss: 0.6981419668197631
Epoch 19, Loss: 0.6626630034446717
Epoch 20, Loss: 0.653494784116745
Epoch 21, Loss: 0.6192986342906952
Epoch 22, Loss: 0.6069417457580566
Epoch 23, Loss: 0.5996500818729401
Epoch 24, Loss: 0.5852193846702576
Epoch 25, Loss: 0.5884738006591796
Epoch 26, Loss: 0.5571978559494019
Epoch 27, Loss: 0.5479833743572236
Epoch 28, Loss: 0.5261419155597686
Epoch 29, Loss: 0.4929752392768859

In [ ]:
user_id = 10
top_items, scores = top_n_anime_for_user(tuned_model, df, user_id, num_items, N=5)

print(f"Top-5 рекомендаций для пользователя {user_id}:")
for item, score in zip(top_items, scores):
    print(f"Аниме ID: {item}, предсказанный рейтинг: {score:.2f}")

Top-5 рекомендаций для пользователя 10:
Аниме ID: 2634, предсказанный рейтинг: 12.88
Аниме ID: 2755, предсказанный рейтинг: 11.96
Аниме ID: 563, предсказанный рейтинг: 11.77
Аниме ID: 642, предсказанный рейтинг: 11.73
Аниме ID: 2203, предсказанный рейтинг: 11.51


## Сравнение моделей

### Базовая модель

**Обучение:**

- Loss стабильно убывает до 0.06
- Модель быстро подстраивается под обучающие данные

**Качество на тесте:**
- RMSE = 1.6624
- MAE = 1.2650

**Наблюдения:**
- Модель показывает неплохое качество, но есть признаки переобучения: train-loss заметно ниже, чем ошибка на тесте
- Рекомендации формируются, но модель может переоценивать популярные объекты

### Улучшенная модель (тюнинг гиперпараметров + регуляризация)

**Изменения по сравнению с базовой моделью:**

- Увеличена размерность эмбеддингов (64)

- Добавлен Dropout внутри полносвязных слоев


**Обучение:**

- Loss убывает более плавно и стабильно
- Процесс обучения более устойчивый

**Качество на тесте:**
- RMSE = 1.4923
- MAE = 1.1429

**Улучшение качества:**
- RMSE снизился с 1.66 до 1.49 (примерно на 10%)
- MAE снизился с 1.27 до 1.14

### Интерпретация результатов

1. **Снижение RMSE и MAE говорит об улучшении качества рекомендаций.**
Модель после тюнинга лучше обобщает знания и точнее предсказывает рейтинги на данных, которые не видела во время обучения.

2. **Регуляризация помогла уменьшить переобучение.**
В базовой модели наблюдался разрыв между ошибкой на обучении и на тесте.
После добавления Dropout этот разрыв сократился, что говорит о лучшей обобщающей способности модели.

3. **Более стабильное обучение.**
В улучшенной версии loss убывает более плавно, что свидетельствует о более удачно подобранных гиперпараметрах оптимизатора и архитектуры.

## Вывод

В ходе работы были проведены эксперименты по улучшению нейросетевой модели рекомендательной системы путём подбора гиперпараметров и добавления регуляризации. По сравнению с базовой моделью, улучшенная конфигурация продемонстрировала более высокое качество на тестовой выборке: значение RMSE снизилось с 1.66 до 1.49, а MAE с 1.27 до 1.14. Это свидетельствует о повышении точности предсказаний и лучшей обобщающей способности модели. Добавление регуляризации позволило уменьшить переобучение, а подбор размерности эмбеддингов обеспечил более стабильный процесс обучения. В результате улучшенная модель формирует более качественные персонализированные рекомендации для пользователей.

Помимо улучшений модели для снижения RMSE и MAE, имеет смысл рассчитывать метрики, ориентированные на качество рекомендаций, такие как Precision@k (доля релевантных объектов среди топ‑k рекомендованных), Recall@k (доля всех релевантных объектов, которые были рекомендованы) и HitRate@k (проверка, попал ли хотя бы один релевантный объект в топ‑k). RMSE и MAE показывают только точность предсказанных рейтингов, но не отражают, насколько рекомендации полезны для пользователя, тогда как эти метрики дают более реалистичную оценку эффективности системы.